1. Parties and Term Checks:

[1] 1.1 The Agreement takes effect on 1 January 2024 and expires on 31 December 2025.

## Scope and AI-assistance disclosure

**Scope:** This notebook develops and evaluates the Hospital 1 audit pipeline only. Hospitals 2–5 are intentionally not processed in this pass because of the exercise time constraint.

**AI assistance:** Claude was used to review the audit logic, suggest edge-case handling, and help refactor/debug the notebook. The final contract interpretations, thresholds, checks, and implementation decisions were reviewed and selected by the author.


In [1]:
from pathlib import Path
import pandas as pd

def find_repo_root(start: Path = None, marker: str = "invoices") -> Path:
    # Works whether the notebook is opened from the repository, /mnt/data, or another Jupyter cwd.
    candidates = []
    start = start or Path.cwd()
    candidates.extend([start, *start.parents])
    candidates.extend([
        Path("/mnt/data/insurance_repo/insurance_auditing-main"),
        Path("/home/claude/work/insurance_auditing-main"),
    ])
    for candidate in candidates:
        if (candidate / marker).is_dir() and (candidate / "contracts").is_dir():
            return candidate.resolve()
    checked = ", ".join(str(c) for c in candidates[:6])
    raise FileNotFoundError(f"Could not find the insurance-auditing repository. Checked: {checked}")

REPO_ROOT = find_repo_root()
print("Repo root:", REPO_ROOT)
print("Working dir was:", Path.cwd())

invoices = pd.read_csv(
    REPO_ROOT / "invoices" / "hospital_1_invoices.csv",
    parse_dates=["invoice_date"],
)
line_items = pd.read_csv(
    REPO_ROOT / "invoices" / "hospital_1_line_items.csv",
    parse_dates=["service_date"],
)


Repo root: /mnt/data/insurance_repo/insurance_auditing-main
Working dir was: /mnt/data


In [2]:

# Work on a copy; do not change the original line_items data
term_check = line_items.copy()

# Contract dates from clause 1.1
contract_start = pd.Timestamp("2024-01-01")
contract_end = pd.Timestamp("2025-12-31")

# Parse dates in the temporary table
term_check["service_date_parsed"] = pd.to_datetime(
    term_check["service_date"],
    format="%Y-%m-%d",
    errors="coerce"
)

# Flag invalid or out-of-term service dates
term_check["term_issue"] = (
    term_check["service_date_parsed"].isna()
    | (term_check["service_date_parsed"] < contract_start)
    | (term_check["service_date_parsed"] > contract_end)
)

print("Service lines checked:", len(term_check))
print("Term issues found:", term_check["term_issue"].sum())

# Display only affected rows
display(
    term_check.loc[
        term_check["term_issue"],
        ["line_id", "invoice_id", "service_date", "service_date_parsed"]
    ]
)



Service lines checked: 11415
Term issues found: 12


,line_id,invoice_id,service_date,service_date_parsed
394,H1-L00036-17,INV-H1-000036,2025-06-31,NaT
830,H1-L00069-02,INV-H1-000069,not-a-date,NaT
1764,H1-L00148-02,INV-H1-000148,2023-06-09,2023-06-09
2157,H1-L00179-06,INV-H1-000179,2026-07-24,2026-07-24
2654,H1-L00219-09,INV-H1-000219,2024-00-17,NaT
3457,H1-L00281-10,INV-H1-000281,31/02/2024,NaT
6876,H1-L00552-12,INV-H1-000552,2025-02-30,NaT
8112,H1-L00650-07,INV-H1-000650,2025-06-31,NaT
8218,H1-L00659-07,INV-H1-000659,2023-02-24,2023-02-24
8220,H1-L00659-09,INV-H1-000659,2022-12-30,2022-12-30


In [3]:
# Check clause 1.2 without changing the original invoices table

facility_check = invoices.copy()

facility_check["facility_issue"] = (
    facility_check["facility_code"].isna()
    | facility_check["facility_code"].ne("F-MAIN")
)

facility_check["facility_finding"] = "Valid Main Campus facility"

facility_check.loc[
    facility_check["facility_code"].isna(),
    "facility_finding"
] = "Missing facility code"

facility_check.loc[
    facility_check["facility_code"].notna()
    & facility_check["facility_code"].ne("F-MAIN"),
    "facility_finding"
] = "Facility is not F-MAIN"

print("Invoice records checked:", len(facility_check))
print("Facility issues found:", facility_check["facility_issue"].sum())

display(
    facility_check.loc[
        facility_check["facility_issue"],
        [
            "invoice_id",
            "facility_code",
            "facility_finding",
            "invoice_total_cents"
        ]
    ]
)

Invoice records checked: 918
Facility issues found: 0


,invoice_id,facility_code,facility_finding,invoice_total_cents


1.3 All patient plan tiers (BRONZE, SILVER, GOLD) are reimbursed at the same rate under this Agreement.



In [4]:
# Check clause 1.3 without changing the original invoices table

plan_check = invoices.copy()

allowed_plans = {"BRONZE", "SILVER", "GOLD"}

plan_check["plan_issue"] = (
    plan_check["plan_tier"].isna()
    | ~plan_check["plan_tier"].isin(allowed_plans)
)

print("Invoice records checked:", len(plan_check))
print("Plan values found:", sorted(plan_check["plan_tier"].dropna().unique()))
print("Invalid or missing plan values:", plan_check["plan_issue"].sum())

display(
    plan_check.loc[
        plan_check["plan_issue"],
        ["invoice_id", "patient_id", "plan_tier"]
    ]
)


Invoice records checked: 918
Plan values found: ['BRONZE', 'GOLD', 'SILVER']
Invalid or missing plan values: 0


,invoice_id,patient_id,plan_tier


(a) substitution of a bundled rate; (b) the facility multiplier; (c) the plan-tier multiplier; (d) any premium or uplift; and (e) any cumulative volume discount.

This creates rate_schedule, a new working table. It does not change the contract file or invoice data.

These are base rates only. The final expected price may later change because of bundles, premiums, discounts, or caps

In [5]:
# Section 4 — extract Hospital 1 base rates

from pathlib import Path
from decimal import Decimal
import re
import pandas as pd
from IPython.display import display

contract_path = REPO_ROOT/ "contracts" / "hospital_1" / "provider_services_agreement.md"
contract_lines = contract_path.read_text(encoding="utf-8").splitlines()

rate_rows = []
inside_section_4 = False

for line_number, line in enumerate(contract_lines, start=1):

    if line.startswith("## 4. Rate Schedule"):
        inside_section_4 = True
        continue

    if inside_section_4 and line.startswith("## 5."):
        break

    if not inside_section_4 or not line.startswith("|"):
        continue

    values = [
        value.strip()
        for value in line.strip("|").split("|")
    ]

    # Skip the table header and separator row
    if values[0] in {"Service", "---"}:
        continue

    if len(values) != 4 or not values[2].startswith("GBP"):
        continue

    service = values[0]
    contract_unit = values[1]
    rate_text = values[2].replace("GBP", "").replace(",", "").strip()
    cap_text = values[3]

    # Convert GBP to integer cents exactly
    base_rate_cents = int(
        Decimal(rate_text) * 100
    )

    # Convert daily cap to an integer where one exists
    daily_cap = (
        int(re.search(r"\d+", cap_text).group())
        if re.search(r"\d+", cap_text)
        else pd.NA
    )

    rate_rows.append({
        "service": service,
        "contract_unit": contract_unit,
        "base_rate_cents": base_rate_cents,
        "daily_cap": daily_cap,
        "contract_section": "4",
        "source_line": line_number
    })

rate_schedule = pd.DataFrame(rate_rows)

# Basic validation
assert not rate_schedule.empty
assert rate_schedule["service"].is_unique
assert rate_schedule["base_rate_cents"].gt(0).all()

print("Contract services extracted:", len(rate_schedule))
print("Services with daily caps:",
      rate_schedule["daily_cap"].notna().sum())

display(rate_schedule.head(10))

Contract services extracted: 108
Services with daily caps: 7


,service,contract_unit,base_rate_cents,daily_cap,contract_section,source_line
0,Advanced Cardiac Recovery Room Occupancy,per hour,20000,<NA>,4,47
1,Advanced Haematology Physiotherapy Session,per hour,10150,<NA>,4,48
2,Advanced Infectious Critical Care Occupancy,per day of service,70925,<NA>,4,49
3,Advanced Metabolic Anaesthesia Administration,per hour,9200,<NA>,4,50
4,Advanced Metabolic Nursing Observation,per day of service,130125,6,4,51
5,Advanced Neurological Consultation,per visit,14125,<NA>,4,52
6,Advanced Rheumatologic Laboratory Panel,per test,14775,4,4,53
7,Ambulatory Cardiac Home Visit,per visit,28425,<NA>,4,54
8,Ambulatory Immunologic Endoscopic Procedure,per procedure,607900,<NA>,4,55
9,Ambulatory Immunologic Ward Bed Occupancy,per day of service,87250,<NA>,4,56


In [6]:
# Sections 5 and 6 — extract premiums and weekend uplifts

import re
import pandas as pd
from IPython.display import display

contract_text = contract_path.read_text(encoding="utf-8")
contract_lines = contract_text.splitlines()

threshold_rows = []
weekend_rows = []
section = None

for line_number, line in enumerate(contract_lines, start=1):

    if line.startswith("## 5. Threshold Premiums"):
        section = 5
        continue

    if line.startswith("## 6. Non-Business-Day Uplifts"):
        section = 6
        continue

    if line.startswith("## 7."):
        section = None

    if not line.startswith("|") or section is None:
        continue

    values = [
        value.strip()
        for value in line.strip("|").split("|")
    ]

    # Section 5
    if section == 5 and len(values) == 3:
        if values[0] in {"Service", "---"}:
            continue

        threshold_match = re.search(r"\d+", values[1])
        uplift_match = re.search(r"\d+", values[2])

        if threshold_match and uplift_match:
            threshold_rows.append({
                "service": values[0],
                "threshold_units": int(threshold_match.group()),
                "premium_percent": int(uplift_match.group()),
                "contract_section": "5",
                "source_line": line_number
            })

    # Section 6
    if section == 6 and len(values) == 2:
        if values[0] in {"Service", "---"}:
            continue

        uplift_match = re.search(r"\d+", values[1])

        if uplift_match:
            weekend_rows.append({
                "service": values[0],
                "weekend_uplift_percent": int(uplift_match.group()),
                "contract_section": "6",
                "source_line": line_number
            })

threshold_premiums = pd.DataFrame(threshold_rows)
weekend_uplifts = pd.DataFrame(weekend_rows)

assert len(threshold_premiums) == 9
assert len(weekend_uplifts) == 7

print("Threshold premium rules:", len(threshold_premiums))
print("Weekend uplift rules:", len(weekend_uplifts))

display(threshold_premiums)
display(weekend_uplifts)

Threshold premium rules: 9
Weekend uplift rules: 7


,service,threshold_units,premium_percent,contract_section,source_line
0,Ambulatory Ophthalmic Case Conference,6,20,5,160
1,Ambulatory Ophthalmic Dialysis Session,10,20,5,161
2,Continuous Musculoskeletal Wound Care,8,40,5,162
3,Emergency Dermatologic Case Conference,10,40,5,163
4,Preoperative Geriatric Ventilation Support,8,20,5,164
5,Preoperative Renal Wound Care,8,25,5,165
6,Routine Psychiatric Rehabilitation Programme,8,25,5,166
7,Specialist Neurological Recovery Room Occupancy,8,30,5,167
8,Standard Pulmonary Dialysis Session,8,25,5,168


,service,weekend_uplift_percent,contract_section,source_line
0,Advanced Neurological Consultation,20,6,176
1,Assisted Geriatric Infusion Therapy,12,6,177
2,Assisted Infectious Discharge Planning,12,6,178
3,Emergency Renal Radiotherapy Fraction,20,6,179
4,Focused Orthopaedic Transport Service,10,6,180
5,Specialist Dermatologic Transport Service,12,6,181
6,Supervised Musculoskeletal Dialysis Session,12,6,182


In [7]:
# Sections 7–9 — extract discounts, caps and bundles

from decimal import Decimal
import re
import pandas as pd
from IPython.display import display

contract_lines = contract_path.read_text(
    encoding="utf-8"
).splitlines()

discount_rows = []
cap_rows = []
bundle_rows = []
section = None

for line_number, line in enumerate(contract_lines, start=1):

    if line.startswith("## 7. Cumulative Volume Discounts"):
        section = 7
        continue

    if line.startswith("## 8. Daily Quantity Caps"):
        section = 8
        continue

    if line.startswith("## 9. Bundled Services"):
        section = 9
        continue

    if line.startswith("## 10."):
        section = None

    if not line.startswith("|") or section is None:
        continue

    values = [
        value.strip()
        for value in line.strip("|").split("|")
    ]

    # Section 7 — cumulative discounts
    if section == 7 and len(values) == 3:
        if values[0] in {"Service", "---"}:
            continue

        threshold = re.search(r"\d+", values[1])
        discount = re.search(r"\d+", values[2])

        if threshold and discount:
            discount_rows.append({
                "service": values[0],
                "threshold_units": int(threshold.group()),
                "discount_percent": int(discount.group()),
                "contract_section": "7",
                "source_line": line_number
            })

    # Section 8 — daily quantity caps
    elif section == 8 and len(values) == 2:
        if values[0] in {"Service", "---"}:
            continue

        cap = re.search(r"\d+", values[1])

        if cap:
            cap_rows.append({
                "service": values[0],
                "maximum_units_per_patient_day": int(cap.group()),
                "contract_section": "8",
                "source_line": line_number
            })

    # Section 9 — bundled rates
    elif section == 9 and len(values) == 4:
        if values[0] in {"Service A", "---"}:
            continue

        rate_a = int(
            Decimal(
                values[2].replace("GBP", "").replace(",", "").strip()
            ) * 100
        )

        rate_b = int(
            Decimal(
                values[3].replace("GBP", "").replace(",", "").strip()
            ) * 100
        )

        bundle_rows.append({
            "service_a": values[0],
            "service_b": values[1],
            "bundled_rate_a_cents": rate_a,
            "bundled_rate_b_cents": rate_b,
            "contract_section": "9",
            "source_line": line_number
        })

volume_discounts = pd.DataFrame(discount_rows)
daily_caps = pd.DataFrame(cap_rows)
bundles = pd.DataFrame(bundle_rows)

# Basic validation against the agreement
assert len(volume_discounts) == 11
assert len(daily_caps) == 7
assert len(bundles) == 3

print("Volume discount rules:", len(volume_discounts))
print("Daily quantity caps:", len(daily_caps))
print("Bundled service pairs:", len(bundles))

display(volume_discounts)
display(daily_caps)
display(bundles)

Volume discount rules: 11
Daily quantity caps: 7
Bundled service pairs: 3


,service,threshold_units,discount_percent,contract_section,source_line
0,Ambulatory Pulmonary Recovery Room Occupancy,60,10,7,188
1,Comprehensive Infectious Nursing Observation,80,10,7,189
2,Comprehensive Infectious Nursing Observation,240,25,7,190
3,Extended Geriatric Wound Care,60,15,7,191
4,Intensive Gastrointestinal Isolation Room Occu...,60,12,7,192
5,Intensive Gastrointestinal Isolation Room Occu...,180,30,7,193
6,Intermittent Pulmonary Rehabilitation Programme,120,10,7,194
7,Preoperative Immunologic Endoscopic Procedure,80,12,7,195
8,Preoperative Immunologic Endoscopic Procedure,240,20,7,196
9,Standard Otolaryngologic Radiotherapy Fraction,60,12,7,197


,service,maximum_units_per_patient_day,contract_section,source_line
0,Advanced Metabolic Nursing Observation,6,8,207
1,Advanced Rheumatologic Laboratory Panel,4,8,208
2,Comprehensive Oncology Nursing Observation,12,8,209
3,Inpatient Palliative Specimen Analysis,4,8,210
4,Routine Infectious Critical Care Occupancy,6,8,211
5,Routine Urologic Biopsy Procedure,6,8,212
6,Standard Endocrine Dialysis Session,8,8,213


,service_a,service_b,bundled_rate_a_cents,bundled_rate_b_cents,contract_section,source_line
0,Advanced Cardiac Recovery Room Occupancy,Routine Cardiac Specimen Analysis,16400,19150,9,219
1,Extended Palliative Laboratory Panel,Inpatient Ophthalmic Radiotherapy Fraction,15175,21500,9,220
2,Inpatient Hepatic Physiotherapy Session,Specialist Otolaryngologic Theatre Time,37500,7050,9,221


## Roadmap for Hospital 1

**Already completed:**

* Loaded Hospital 1 invoices and line items.
* Checked the contract term from clause 1.1.
* Checked facility code from clause 1.2.
* Checked plan tiers from clause 1.3.
* Extracted Section 4 base rates.
* Extracted Sections 5–6 premiums and weekend uplifts.
* Extracted Sections 7–9 discounts, caps, and bundles.

**What remains, in this order:**

1. Extract Section 10 exclusion windows.
2. Create service matching. Map billing descriptions to the official Section 4
   service names. Keep exact and unambiguous matches; send ambiguous/unmatched
   descriptions for review.
3. Complete Section 11 checks:
   * contract-number mismatch;
   * reused invoice IDs;
   * invalid/out-of-term/service-date-after-invoice dates;
   * duplicate patient/service/service-date billing.
4. Calculate expected prices using the contract order:
   `bundle → facility → plan → premium → weekend uplift → volume discount → quantity`
5. Aggregate line totals to invoice totals.
6. Compare predictions with Hospital 1 labels.
7. Create the final submission file with:
   `invoice_id, flagged, error_category, expected_total_cents, billed_total_cents, confidence`

The next cell is the service-matching cell — everything downstream (Section 11's
duplicate-billing check, the pricing engine, aggregation) needs `matched_service`
to exist first, so it has to come before the checks and before pricing, not after.

**Ground rules for this pass:**
Do not mark unmatched descriptions as errors automatically — they go to a review
list, because the billing description may still correspond to a valid contracted
service we just didn't recognise. Do not force predictions to match every label —
the goal is reliable precision and honest, calibrated uncertainty, not confident
guesses.


### Step 1 — Section 10: Exclusion Windows

10.1 An exclusion window is measured in either direction from the Service Date
of the excluded Service.

In [8]:
# Section 10 — extract exclusion windows

contract_lines = contract_path.read_text(encoding="utf-8").splitlines()

exclusion_rows = []
section = None

for line_number, line in enumerate(contract_lines, start=1):

    if line.startswith("## 10. Exclusion Windows"):
        section = 10
        continue

    if line.startswith("## 11."):
        section = None

    if not line.startswith("|") or section is None:
        continue

    values = [
        value.strip()
        for value in line.strip("|").split("|")
    ]

    if section == 10 and len(values) == 3:
        if values[0] in {"Service", "---"}:
            continue

        days_match = re.search(r"\d+", values[1])
        if days_match:
            exclusion_rows.append({
                "service_a": values[0],
                "window_days": int(days_match.group()),
                "service_b": values[2],
                "contract_section": "10",
                "source_line": line_number
            })

exclusion_windows = pd.DataFrame(exclusion_rows)

assert len(exclusion_windows) == 6

print("Exclusion window rules:", len(exclusion_windows))
display(exclusion_windows)


Exclusion window rules: 6


,service_a,window_days,service_b,contract_section,source_line
0,Advanced Metabolic Anaesthesia Administration,7,Standard Endocrine Endoscopic Procedure,10,229
1,Continuous Immunologic Theatre Time,21,Supervised Otolaryngologic Sterilisation Service,10,230
2,Intensive Ophthalmic Case Conference,7,Continuous Otolaryngologic Telemetry Monitoring,10,231
3,Postoperative Ophthalmic Radiotherapy Fraction,30,Inpatient Palliative Isolation Room Occupancy,10,232
4,Routine Immunologic Ward Bed Occupancy,10,Comprehensive Otolaryngologic Theatre Time,10,233
5,Standard Paediatric Biopsy Procedure,10,Advanced Infectious Critical Care Occupancy,10,234


### Step 2 — Service matching

`description` is free text and heavily abbreviated. The matcher is deliberately conservative because a wrong service mapping contaminates every downstream rate check.

**Matching policy**

1. Normalize common billing abbreviations and strip trailing reference codes.
2. Compare candidate services using token-level exact, prefix, subsequence and fuzzy evidence.
3. Apply two structural constraints before accepting a match:
   - a clinical-specialty anchor in the description must agree with the candidate specialty;
   - a specific service-class anchor (for example `specimen/analysis`, `dialysis`, `wound care`, `consultation`, `transport`, `occupancy`) must be represented by the candidate when present.
4. Accept only a sufficiently strong, clearly separated top candidate.
5. When the text is genuinely tied, billed price may be used **only as secondary evidence** among structurally compatible candidates, and only when exactly one candidate has a plausible contract price.
6. Everything else becomes `needs_review`. A review case is an audit flag with low confidence; it is not silently treated as a correct invoice.

This keeps the audit conservative without using the billed amount as the primary definition of the service being audited.

In [9]:
# Service-matching functions — conservative, reusable, and price only as secondary evidence

import difflib
from decimal import Decimal, ROUND_HALF_UP

REF_CODE_RE = re.compile(r"/[A-Z]{1,4}-?\d+\s*$")

TOKEN_ALIASES = {
    # Generic billing abbreviations
    "adv": "advanced", "asst": "assisted", "amb": "ambulatory", "inpt": "inpatient",
    "outpt": "outpatient", "obs": "observation", "spclst": "specialist", "supv": "supervised",
    "rtn": "routine", "std": "standard", "cont": "continuous", "interm": "intermittent",
    "proc": "procedure", "procure": "procedure", "sess": "session", "svc": "service",
    "prog": "programme", "physio": "physiotherapy", "occ": "occupancy", "rm": "room",
    "cs": "case", "conf": "conference", "wd": "ward", "bd": "bed", "anaes": "anaesthesia",
    "anaest": "anaesthesia", "anest": "anaesthesia", "admin": "administration",
    "spcm": "specimen", "anly": "analysis", "pnl": "panel", "plng": "planning",
    "disp": "dispensing", "pharm": "pharmaceutical", "radiother": "radiotherapy",
    "radioth": "radiotherapy", "biop": "biopsy", "dial": "dialysis", "transf": "transfusion",
    "crit": "critical", "cr": "critical", "nutr": "nutritional", "wnd": "wound",
    "rehab": "rehabilitation", "vent": "ventilation", "tele": "telemetry", "img": "imaging",
    "diag": "diagnostic", "isom": "isolation", "paed": "paediatric", "paeds": "paediatric",
    # Clinical / specialty abbreviations
    "ent": "otolaryngologic", "gi": "gastrointestinal", "ortho": "orthopaedic",
    "ophth": "ophthalmic", "haem": "haematology", "hema": "haematology", "neuro": "neurological",
    "rheum": "rheumatologic", "urol": "urologic", "pulm": "pulmonary", "psych": "psychiatric",
    "ger": "geriatric", "onco": "oncology", "derm": "dermatologic", "hep": "hepatic",
    "vasc": "vascular", "endo": "endocrine", "obst": "obstetric", "immun": "immunologic",
    "infect": "infectious", "msk": "musculoskeletal", "musk": "musculoskeletal", "pall": "palliative",
    "card": "cardiac", "metab": "metabolic",
}

CLINICAL_ANCHORS = {
    "cardiac", "haematology", "infectious", "metabolic", "neurological", "rheumatologic",
    "ophthalmic", "immunologic", "musculoskeletal", "gastrointestinal", "geriatric", "urologic",
    "pulmonary", "psychiatric", "oncology", "otolaryngologic", "palliative", "renal", "vascular",
    "dermatologic", "paediatric", "endocrine", "obstetric", "hepatic", "orthopaedic"
}

SERVICE_CLASS_ALIASES = {
    "specimen": {"specimen", "analysis", "panel", "laboratory", "lab"},
    "analysis": {"specimen", "analysis", "panel", "laboratory", "lab"},
    "panel": {"specimen", "analysis", "panel", "laboratory", "lab"},
    "dialysis": {"dialysis"},
    "wound": {"wound", "care"},
    "rehabilitation": {"rehabilitation", "programme"},
    "consultation": {"consultation"},
    "transport": {"transport"},
    "occupancy": {"occupancy", "room", "bed", "ward"},
    "endoscopic": {"endoscopic", "procedure"},
    "radiotherapy": {"radiotherapy", "fraction"},
    "infusion": {"infusion", "therapy"},
    "discharge": {"discharge", "planning"},
    "pharmaceutical": {"pharmaceutical", "dispensing"},
    "ventilation": {"ventilation", "support"},
    "nutritional": {"nutritional", "support"},
    "transfusion": {"transfusion"},
    "sterilisation": {"sterilisation"},
    "theatre": {"theatre"},
    "critical": {"critical", "care"},
}

def normalize_description(text: str):
    text = REF_CODE_RE.sub("", str(text))
    text = text.replace("-", " ").replace("/", " ")
    tokens = re.findall(r"[A-Za-z]+", text.lower())
    return [TOKEN_ALIASES.get(t, t) for t in tokens]

def word_score(token: str, word: str) -> float:
    token, word = token.lower(), word.lower()
    if token == word:
        return 1.0
    if len(token) >= 3 and word.startswith(token):
        return 0.85 + 0.15 * (len(token) / len(word))
    it = iter(word)
    if len(token) >= 2 and all(ch in it for ch in token):
        return 0.45 + 0.35 * (len(token) / len(word))
    r = difflib.SequenceMatcher(None, token, word).ratio()
    return r * 0.55 if r > 0.75 else 0.0

def score_candidate(desc_tokens, service_words):
    pairs = sorted(
        ((word_score(t, w), i, j) for i, t in enumerate(desc_tokens) for j, w in enumerate(service_words)),
        reverse=True,
    )
    used_i, used_j, matched_score, matched_pairs = set(), set(), 0.0, 0
    for s, i, j in pairs:
        if s <= 0 or i in used_i or j in used_j:
            continue
        used_i.add(i); used_j.add(j)
        matched_score += s
        matched_pairs += 1
    n_words, n_tokens = len(service_words), len(desc_tokens)
    coverage_desc = matched_pairs / n_tokens if n_tokens else 0
    return (matched_score / n_words) * (0.6 + 0.4 * coverage_desc) if n_words else 0

def anchor_sets(tokens):
    anchors = {t for t in tokens if t in CLINICAL_ANCHORS}
    classes = set()
    token_set = set(tokens)
    for key, aliases in SERVICE_CLASS_ALIASES.items():
        if token_set & aliases:
            classes.add(key)
    return anchors, classes

def candidate_conflict(desc_tokens, service_tokens):
    d_anchor, d_class = anchor_sets(desc_tokens)
    s_anchor, s_class = anchor_sets(service_tokens)
    anchor_conflict = bool(d_anchor and s_anchor and d_anchor.isdisjoint(s_anchor))
    class_conflict = bool(d_class and s_class and d_class.isdisjoint(s_class))
    return anchor_conflict, class_conflict, d_anchor, d_class, s_anchor, s_class

def match_description(desc: str, schedule: pd.DataFrame, top_k=8):
    tokens = normalize_description(desc)
    results = []
    for _, row in schedule.iterrows():
        svc = row["service"]
        svc_tokens = normalize_description(svc)
        score = score_candidate(tokens, svc_tokens)
        anchor_conflict, class_conflict, d_anchor, d_class, s_anchor, s_class = candidate_conflict(tokens, svc_tokens)
        if anchor_conflict:
            score -= 0.40
        if class_conflict:
            score -= 0.20
        results.append({
            "score": score,
            "service": svc,
            "anchor_conflict": anchor_conflict,
            "class_conflict": class_conflict,
            "d_anchor": d_anchor, "d_class": d_class,
            "s_anchor": s_anchor, "s_class": s_class,
        })
    results.sort(key=lambda r: (r["score"], r["service"]), reverse=True)
    return results[:top_k]

GAP_THRESHOLD = 0.08
MIN_SCORE = 0.55

def plausible_unit_prices(service):
    base = int(rate_schedule.loc[rate_schedule["service"] == service, "base_rate_cents"].iloc[0])
    prices = {base}
    for _, r in threshold_premiums.loc[threshold_premiums["service"] == service].iterrows():
        prices.add(half_up(Decimal(base) * (Decimal(100 + int(r["premium_percent"])) / 100)))
    for _, r in weekend_uplifts.loc[weekend_uplifts["service"] == service].iterrows():
        prices.add(half_up(Decimal(base) * (Decimal(100 + int(r["weekend_uplift_percent"])) / 100)))
    for _, r in volume_discounts.loc[volume_discounts["service"] == service].iterrows():
        prices.add(half_up(Decimal(base) * (Decimal(100 - int(r["discount_percent"])) / 100)))
    return prices

def half_up(value):
    return int(Decimal(value).to_integral_value(rounding=ROUND_HALF_UP))


In [10]:
# Resolve descriptions, then assign line groups to invoice occurrences.
# The occurrence mapping prevents reused invoice IDs from doubling line items.

from IPython.display import display

# Preserve original line-group identity. Synthetic Hospital 1 line IDs retain the
# transaction group even when an invoice identifier is accidentally reused.
line_items["line_group_num"] = (
    line_items["line_id"].astype(str)
    .str.extract(r"^H1-L(\d+)-", expand=False)
    .astype("Int64")
)

if line_items["line_group_num"].isna().any():
    raise ValueError("Unable to recover line-group identity from one or more line IDs")

# Invoice occurrence rank: chronological within reused invoice IDs.
invoices["_row_order"] = range(len(invoices))
invoices["occurrence_rank"] = (
    invoices.sort_values(["invoice_id", "invoice_date", "_row_order"])
    .groupby("invoice_id").cumcount()
)

# Line-group rank: chronological source-group order.
group_meta = (
    line_items.groupby(["invoice_id", "line_group_num"], dropna=False)
    .agg(group_billed_total_cents=("line_total_cents", "sum"),
         group_min_service_raw=("service_date", "first"))
    .reset_index()
)
group_meta["group_rank"] = (
    group_meta.sort_values(["invoice_id", "line_group_num"])
    .groupby("invoice_id").cumcount()
)

# Map each group rank to the invoice occurrence rank. Non-duplicate IDs simply get 0.
line_items = line_items.merge(
    group_meta[["invoice_id", "line_group_num", "group_rank", "group_billed_total_cents"]],
    on=["invoice_id", "line_group_num"], how="left", suffixes=("", "_group")
)
line_items["occurrence_rank"] = line_items["group_rank"].astype(int)

occ_cols = ["invoice_id", "occurrence_rank", "invoice_date", "patient_id",
            "facility_code", "plan_tier", "contract_number", "invoice_total_cents"]
line_items = line_items.merge(
    invoices[occ_cols], on=["invoice_id", "occurrence_rank"], how="left",
    validate="many_to_one", suffixes=("", "_invoice")
)

# Validate reused-ID mapping against billed totals when possible.
occ_validation = (
    line_items.groupby(["invoice_id", "occurrence_rank"], as_index=False)
    .agg(group_lines_billed_cents=("line_total_cents", "sum"))
    .merge(invoices[occ_cols], on=["invoice_id", "occurrence_rank"], how="left")
)
occ_validation["billed_group_matches_invoice"] = (
    occ_validation["group_lines_billed_cents"] == occ_validation["invoice_total_cents"]
)

# Per-description matching. No billed-price tie-break is allowed unless text/structure
# leaves compatible candidates and exactly one has a plausible observed unit price.
observed_prices = (
    line_items.groupby("description")["unit_price_cents"]
    .apply(lambda s: set(int(x) for x in s.unique()))
)

match_records = []
for desc in line_items["description"].dropna().astype(str).unique():
    top = match_description(desc, rate_schedule, top_k=8)
    top1, top2 = top[0], top[1] if len(top) > 1 else None
    gap = top1["score"] - (top2["score"] if top2 else 0.0)
    obs = observed_prices.get(desc, set())

    structurally_compatible = [
        r for r in top
        if not r["anchor_conflict"] and not r["class_conflict"]
    ]

    status = "needs_review"
    chosen = None
    reason = ""

    # Strong, separated text match.
    if (not top1["anchor_conflict"] and not top1["class_conflict"]
            and top1["score"] >= MIN_SCORE and gap >= GAP_THRESHOLD):
        status, chosen = "matched", top1["service"]
        reason = "strong_text_match"
    else:
        # Secondary price evidence is allowed only among structurally compatible candidates.
        close = [r for r in structurally_compatible if r["score"] >= max(top1["score"] - GAP_THRESHOLD, 0)]
        consistent = [r for r in close if obs and obs.issubset(plausible_unit_prices(r["service"]))]
        if len(consistent) == 1 and consistent[0]["score"] >= 0.35:
            status, chosen = "resolved_by_price", consistent[0]["service"]
            reason = "secondary_price_support"
        else:
            if top1["anchor_conflict"]:
                reason = "clinical_anchor_conflict"
            elif top1["class_conflict"]:
                reason = "service_class_conflict"
            elif len(structurally_compatible) > 1:
                reason = "ambiguous_text_match"
            else:
                reason = "insufficient_text_evidence"

    match_records.append({
        "description": desc,
        "matched_service": chosen,
        "text_score": round(top1["score"], 3),
        "gap": round(gap, 3),
        "match_status": status,
        "match_reason": reason,
        "top_candidates": str([(round(r["score"], 3), r["service"]) for r in top[:3]]),
    })

description_matches = pd.DataFrame(match_records)
print("Match status breakdown:")
print(description_matches["match_status"].value_counts())

descriptions_for_review = description_matches[description_matches["match_status"] == "needs_review"]
print(f"\n{len(descriptions_for_review)} descriptions sent for manual review:")
display(descriptions_for_review[["description", "text_score", "gap", "match_reason", "top_candidates"]])

# Attach matching to line items.
line_items = line_items.merge(
    description_matches[["description", "matched_service", "match_status", "match_reason"]],
    on="description", how="left", validate="many_to_one"
)

line_items["service_date_raw"] = line_items["service_date"]
line_items["service_date"] = pd.to_datetime(
    line_items["service_date_raw"], format="%Y-%m-%d", errors="coerce"
)

# A stable occurrence key is useful for aggregation and debugging.
line_items["invoice_occurrence_key"] = (
    line_items["invoice_id"].astype(str) + "::" + line_items["occurrence_rank"].astype(str)
)

print("\nLine items by match status:")
print(line_items["match_status"].value_counts())
print("\nDuplicate-ID occurrence groups whose billed line sum exactly matches the invoice record:",
      int(occ_validation["billed_group_matches_invoice"].sum()), "/", len(occ_validation))


Match status breakdown:
match_status
matched              462
resolved_by_price     15
needs_review          11
Name: count, dtype: int64

11 descriptions sent for manual review:


,description,text_score,gap,match_reason,top_candidates
255,Extended Haem Anaes Admin,0.175,0.000,ambiguous_text_match,"[(0.175, 'Routine Haematology Infusion Therapy..."
439,Session Interm Hep Dial,0.400,0.000,ambiguous_text_match,"[(0.4, 'Specialist Hepatic Physiotherapy Sessi..."
465,Amb Gastrointestinal Discharge Plng,0.000,0.000,clinical_anchor_conflict,"[(0.0, 'Routine Oncology Discharge Planning'),..."
472,Cont Pulm Spcm Analysis,0.200,0.200,service_class_conflict,"[(0.2, 'Continuous Pulmonary Wound Care'), (0...."
480,Extended Ortho Ventilation Support,0.033,0.033,service_class_conflict,"[(0.033, 'Emergency Orthopaedic Consultation')..."
481,Outpatient Orthopaedic Pharm Dispensing /NG-2825,0.033,0.033,service_class_conflict,"[(0.033, 'Emergency Orthopaedic Consultation')..."
483,Supp Postoperative Ortho Nutritional,0.033,0.046,service_class_conflict,"[(0.033, 'Emergency Orthopaedic Consultation')..."
484,Occ Advanced Orthopaedic Recovery Rm /NG-2920,0.336,0.232,clinical_anchor_conflict,"[(0.336, 'Advanced Cardiac Recovery Room Occup..."
485,Metabolic - Lab Pnl,0.183,0.000,ambiguous_text_match,"[(0.183, 'Advanced Metabolic Nursing Observati..."
486,Adv Renal Consultation,0.178,0.194,clinical_anchor_conflict,"[(0.178, 'Advanced Neurological Consultation')..."



Line items by match status:
match_status
matched              11072
resolved_by_price      332
needs_review            11
Name: count, dtype: int64

Duplicate-ID occurrence groups whose billed line sum exactly matches the invoice record: 912 / 918


### Step 3 — Section 11 checks

* **11.1** contract-number mismatch
* **11.2** reused invoice IDs
* **11.3** invalid / out-of-term / service-date-after-invoice-date
* **11.4** duplicate patient/service/service-date billing (one invoice or across several)

Two additional contract constraints are checked before pricing: **daily quantity caps** (Section 8) and **billed unit basis vs. contract unit** (Section 4).

For reused invoice identifiers, the line IDs preserve separate transaction groups. The notebook assigns those groups to the invoice occurrences chronologically and validates the mapping against each occurrence's billed total where possible. The development labels consistently identify the later occurrence as the erroneous duplicate; that convention is used for the one-row-per-invoice prediction.

For exclusion windows, Section 10 defines `service_a` as the service that is **not billable** when it occurs within the specified window of `service_b`. Therefore the excluded `service_a` line is flagged and removed from the expected reimbursement, rather than blaming whichever invoice happened to be billed later.

In [11]:
# Section 11 checks + unit basis, daily cap, duplicate and exclusion checks

CONTRACT_NUMBER = "INS-H1-2024-0417"
CONTRACT_START = pd.Timestamp("2024-01-01")
CONTRACT_END = pd.Timestamp("2025-12-31")

# --- 11.1 / 11.2 invoice-level checks ---
invoices["contract_number_mismatch"] = invoices["contract_number"] != CONTRACT_NUMBER
invoices["duplicate_invoice_id"] = invoices["invoice_id"].duplicated(keep=False)

# --- 11.3 service dates ---
line_items["malformed_service_date"] = line_items["service_date"].isna()
line_items["service_date_out_of_window"] = (
    (~line_items["malformed_service_date"])
    & ((line_items["service_date"] < CONTRACT_START) | (line_items["service_date"] > CONTRACT_END))
)
line_items["service_date_after_invoice_date"] = (
    (~line_items["malformed_service_date"])
    & (line_items["service_date"] > line_items["invoice_date"])
)

# --- billed unit basis vs contract unit ---
CONTRACT_UNIT_TO_BILLED = {
    "per hour": "per_hour", "per day of service": "per_day", "per visit": "per_visit",
    "per test": "per_test", "per procedure": "per_procedure", "per night of occupancy": "per_night",
    "per item supplied": "per_item", "per unit dispensed": "per_unit_dispensed",
}
_unit_map = rate_schedule.set_index("service")["contract_unit"].map(CONTRACT_UNIT_TO_BILLED).to_dict()
line_items["expected_unit_basis"] = line_items["matched_service"].map(_unit_map)
line_items["wrong_unit_basis"] = (
    line_items["expected_unit_basis"].notna()
    & (line_items["expected_unit_basis"] != line_items["unit_basis_as_billed"])
)

# --- daily caps: cap is applied across all lines for patient+service+Service Day ---
_cap_map = daily_caps.set_index("service")["maximum_units_per_patient_day"].to_dict()
_valid_priced = line_items.dropna(subset=["matched_service", "service_date", "patient_id"]).copy()
_valid_priced = _valid_priced.sort_values(["patient_id", "matched_service", "service_date", "line_id"])
_valid_priced["_day_qty_prior"] = _valid_priced.groupby(
    ["patient_id", "matched_service", "service_date"]
)["quantity"].cumsum() - _valid_priced["quantity"]
_valid_priced["_cap"] = _valid_priced["matched_service"].map(_cap_map)
_valid_priced["billable_quantity"] = _valid_priced["quantity"]
_mask_capped = _valid_priced["_cap"].notna()
_valid_priced.loc[_mask_capped, "billable_quantity"] = (
    _valid_priced.loc[_mask_capped, "quantity"]
    .where(_valid_priced.loc[_mask_capped, "_day_qty_prior"] < _valid_priced.loc[_mask_capped, "_cap"], 0)
    .clip(upper=_valid_priced.loc[_mask_capped, "_cap"] - _valid_priced.loc[_mask_capped, "_day_qty_prior"])
    .clip(lower=0)
)
line_items["_day_qty_prior"] = pd.NA
line_items["billable_quantity"] = line_items["quantity"]
line_items.loc[_valid_priced.index, "_day_qty_prior"] = _valid_priced["_day_qty_prior"]
line_items.loc[_valid_priced.index, "billable_quantity"] = _valid_priced["billable_quantity"]
line_items["daily_cap_exceeded"] = line_items["billable_quantity"] < line_items["quantity"]

# --- 11.4 duplicate patient/service/date billing ---
line_items["cross_invoice_duplicate"] = False
_dup_valid = _valid_priced[_valid_priced["matched_service"].notna()].copy()
for _, grp in _dup_valid.groupby(["patient_id", "matched_service", "service_date"], dropna=False):
    if len(grp) < 2:
        continue
    ordered = grp.sort_values(["invoice_date", "line_id"])
    line_items.loc[ordered.index[1:], "cross_invoice_duplicate"] = True

# --- Section 10 exclusion windows ---
line_items["exclusion_violation"] = False
for _, rule in exclusion_windows.iterrows():
    a_rows = _valid_priced[_valid_priced["matched_service"] == rule["service_a"]]
    b_rows = _valid_priced[_valid_priced["matched_service"] == rule["service_b"]]
    if a_rows.empty or b_rows.empty:
        continue
    merged = a_rows[["patient_id", "service_date"]].reset_index().merge(
        b_rows[["patient_id", "service_date"]].reset_index(),
        on="patient_id", suffixes=("_a", "_b"),
    )
    diff_days = (merged["service_date_a"] - merged["service_date_b"]).abs().dt.days
    violations = merged[diff_days <= int(rule["window_days"])].copy()
    # service_a is the excluded/non-billable service according to Section 10.
    if not violations.empty:
        line_items.loc[violations["index_a"], "exclusion_violation"] = True

# --- billed line arithmetic, an explicit audit check independent of contract pricing ---
line_items["line_total_arithmetic"] = (
    line_items["line_total_cents"] != line_items["unit_price_cents"] * line_items["quantity"]
)

LINE_CHECK_COLS = [
    "malformed_service_date", "service_date_out_of_window", "service_date_after_invoice_date",
    "wrong_unit_basis", "daily_cap_exceeded", "cross_invoice_duplicate", "exclusion_violation",
    "line_total_arithmetic"
]

for col in LINE_CHECK_COLS:
    print(f"{col}: {int(line_items[col].sum())} line items")
print("\ncontract_number_mismatch:", int(invoices["contract_number_mismatch"].sum()), "invoice rows")
print("duplicate_invoice_id:", int(invoices["duplicate_invoice_id"].sum()), "invoice rows")


malformed_service_date: 6 line items
service_date_out_of_window: 6 line items
service_date_after_invoice_date: 9 line items
wrong_unit_basis: 12 line items
daily_cap_exceeded: 4 line items
cross_invoice_duplicate: 4 line items
exclusion_violation: 4 line items
line_total_arithmetic: 6 line items

contract_number_mismatch: 5 invoice rows
duplicate_invoice_id: 10 invoice rows


### Step 4 — Calculate expected prices

The pricing engine follows clause 3.2 exactly: **bundle → facility → plan → premium/uplift → cumulative volume discount → quantity**. Intermediate monetary steps use integer cents and half-up rounding.

Hospital 1 has a single facility and identical plan-tier reimbursement, so those two multipliers are 1×.

Daily caps are applied to the **billable quantity** across all lines for the same Patient + Service + Service Day, allocating the cap in line-ID order. The cap affects the corrected expected total but does not change the contractual unit rate.

An unmatched service or a malformed service date prevents a defensible contract calculation for that line. Instead of dropping the amount or inventing a rate, the notebook carries the billed line total through as a **provisional** expected amount and lowers confidence. That keeps the invoice total numerically complete while making the uncertainty explicit.

Threshold premiums use the contract's stated aggregate daily quantity trigger. Volume discounts use cumulative hospital-wide utilisation and the contract's prior-to-line threshold rule.

In [12]:
# Pricing engine: bundle -> facility -> plan -> premium -> weekend uplift -> volume discount -> quantity

base_rate_map = rate_schedule.set_index("service")["base_rate_cents"].to_dict()
line_items["base_rate_cents"] = line_items["matched_service"].map(base_rate_map)

# --- (a) bundle substitution ---
bundle_map_a = {row.service_a: (row.service_b, row.bundled_rate_a_cents) for row in bundles.itertuples()}
bundle_map_b = {row.service_b: (row.service_a, row.bundled_rate_b_cents) for row in bundles.itertuples()}

_services_present = (
    line_items.dropna(subset=["matched_service", "patient_id", "service_date"])
    .groupby(["patient_id", "service_date"])["matched_service"]
    .apply(set)
)

def _bundle_rate(row):
    svc, key = row["matched_service"], (row["patient_id"], row["service_date"])
    present = _services_present.get(key, set())
    if svc in bundle_map_a and bundle_map_a[svc][0] in present:
        return bundle_map_a[svc][1]
    if svc in bundle_map_b and bundle_map_b[svc][0] in present:
        return bundle_map_b[svc][1]
    return row["base_rate_cents"]

line_items["rate_after_bundle"] = line_items.apply(_bundle_rate, axis=1)

# --- (b) facility / (c) plan-tier multipliers: both 1.0x for Hospital 1 ---
line_items["rate_after_facility_plan"] = line_items["rate_after_bundle"]

# --- (d.1) threshold premium, based on aggregate daily quantity ---
_premium_map = threshold_premiums.set_index("service")[["threshold_units", "premium_percent"]].to_dict("index")
line_items["_day_qty_this_service"] = line_items.groupby(
    ["patient_id", "matched_service", "service_date"]
)["quantity"].transform("sum")

line_items["premium_applies"] = False
line_items["rate_after_premium"] = line_items["rate_after_facility_plan"]
for idx, row in line_items.iterrows():
    rule = _premium_map.get(row["matched_service"])
    rate = row["rate_after_facility_plan"]
    if pd.isna(rate) or rule is None or pd.isna(row["_day_qty_this_service"]):
        continue
    if row["_day_qty_this_service"] > rule["threshold_units"]:
        line_items.at[idx, "premium_applies"] = True
        line_items.at[idx, "rate_after_premium"] = half_up(
            Decimal(rate) * (Decimal(100 + int(rule["premium_percent"])) / 100)
        )

# --- (d.2) weekend / non-business-day uplift ---
_weekend_map = weekend_uplifts.set_index("service")["weekend_uplift_percent"].to_dict()
_is_weekend = line_items["service_date"].dt.dayofweek >= 5
line_items["weekend_applies"] = False
line_items["rate_after_weekend"] = line_items["rate_after_premium"]
for idx, (r, s, w) in enumerate(zip(line_items["rate_after_premium"], line_items["matched_service"], _is_weekend)):
    pct = _weekend_map.get(s)
    if pd.isna(r) or pct is None or pd.isna(w) or not w:
        continue
    line_items.at[line_items.index[idx], "weekend_applies"] = True
    line_items.at[line_items.index[idx], "rate_after_weekend"] = half_up(
        Decimal(r) * (Decimal(100 + int(pct)) / 100)
    )

# --- (e) cumulative volume discount: hospital-wide, service_date then line_id ---
line_items["_discount_pct"] = 0
for service, _ in volume_discounts.groupby("service"):
    idx = line_items.index[line_items["matched_service"] == service]
    if len(idx) == 0:
        continue
    sub = line_items.loc[idx].sort_values(["service_date", "line_id"])
    cumulative_before = sub["quantity"].cumsum() - sub["quantity"]
    tiers = volume_discounts[volume_discounts["service"] == service].sort_values("threshold_units")
    pct = pd.Series(0, index=sub.index, dtype="int64")
    for _, tier in tiers.iterrows():
        pct = pct.where(cumulative_before <= int(tier["threshold_units"]), int(tier["discount_percent"]))
    line_items.loc[sub.index, "_discount_pct"] = pct

def _apply_discount(rate, pct):
    if pd.isna(rate) or not pct:
        return rate
    return half_up(Decimal(rate) * (Decimal(100 - int(pct)) / 100))

line_items["effective_unit_rate_cents"] = [
    _apply_discount(r, p)
    for r, p in zip(line_items["rate_after_weekend"], line_items["_discount_pct"])
]

# Keep billed line totals as a provisional amount when the contract basis is not known.
line_items["expected_line_total_provisional"] = False
line_items["expected_line_total_cents"] = (
    line_items["effective_unit_rate_cents"] * line_items["billable_quantity"].astype("float64")
)
_provisional = line_items["matched_service"].isna() | line_items["malformed_service_date"]
line_items.loc[_provisional, "expected_line_total_cents"] = line_items.loc[_provisional, "line_total_cents"]
line_items.loc[_provisional, "expected_line_total_provisional"] = True

# Excluded service_a lines are not billable.
line_items.loc[line_items["exclusion_violation"], "expected_line_total_cents"] = 0
line_items.loc[line_items["exclusion_violation"], "expected_line_total_provisional"] = False

# A later duplicate of the same patient + service + service day is not separately reimbursable.
# Zero it out in the corrected total, while keeping the billed amount visible for audit evidence.
line_items.loc[line_items["cross_invoice_duplicate"], "expected_line_total_cents"] = 0
line_items.loc[line_items["cross_invoice_duplicate"], "expected_line_total_provisional"] = False

# Bundle / premium / discount diagnostic categories compare the billed rate with each stage.
line_items["bundle_applies"] = line_items.apply(
    lambda r: (
        pd.notna(r["matched_service"])
        and (r["matched_service"] in bundle_map_a or r["matched_service"] in bundle_map_b)
        and ((r["patient_id"], r["service_date"]) in _services_present.index)
        and (
            (r["matched_service"] in bundle_map_a and bundle_map_a[r["matched_service"]][0] in _services_present.get((r["patient_id"], r["service_date"]), set()))
            or (r["matched_service"] in bundle_map_b and bundle_map_b[r["matched_service"]][0] in _services_present.get((r["patient_id"], r["service_date"]), set()))
        )
    ), axis=1
)
line_items["bundle_not_applied"] = (
    line_items["bundle_applies"]
    & (line_items["unit_price_cents"] == line_items["base_rate_cents"])
    & (line_items["rate_after_bundle"] != line_items["base_rate_cents"])
)

line_items["unit_price_mismatch"] = (
    line_items["matched_service"].notna()
    & line_items["effective_unit_rate_cents"].notna()
    & (line_items["unit_price_cents"] != line_items["effective_unit_rate_cents"])
)

# Premium diagnostics. A threshold premium is omitted when the billed rate is the pre-premium rate.
_premium_rate_options = {}
for service, rows in threshold_premiums.groupby("service"):
    base = int(base_rate_map[service])
    _premium_rate_options[service] = {
        half_up(Decimal(base) * (Decimal(100 + int(p)) / 100))
        for p in rows["premium_percent"]
    }
line_items["premium_omitted"] = (
    line_items["premium_applies"]
    & (line_items["unit_price_cents"] == line_items["rate_after_facility_plan"])
    & (line_items["unit_price_cents"] != line_items["effective_unit_rate_cents"])
)
line_items["premium_incorrectly_applied"] = line_items.apply(
    lambda r: (
        pd.notna(r["matched_service"])
        and (r["unit_price_cents"] != r["effective_unit_rate_cents"])
        and (
            (bool(r["premium_applies"]) and not bool(r["premium_omitted"])
             and r["unit_price_cents"] in _premium_rate_options.get(r["matched_service"], set()))
            or (not bool(r["premium_applies"])
                and int(r["unit_price_cents"]) in _premium_rate_options.get(r["matched_service"], set()))
        )
    ), axis=1
)

# Weekend uplift diagnostics are tracked separately. The H1 labels use the broader
# premium_omitted / premium_incorrectly_applied vocabulary for this type of uplift,
# while the final prediction keeps the more precise contract wording.
_weekend_rate_options = {}
for service, rows in weekend_uplifts.groupby("service"):
    base = int(base_rate_map[service])
    _weekend_rate_options[service] = {
        half_up(Decimal(base) * (Decimal(100 + int(p)) / 100))
        for p in rows["weekend_uplift_percent"]
    }
line_items["weekend_uplift_omitted"] = (
    line_items["weekend_applies"]
    & (line_items["unit_price_cents"] == line_items["rate_after_premium"])
    & (line_items["unit_price_cents"] != line_items["effective_unit_rate_cents"])
)
line_items["weekend_uplift_incorrectly_applied"] = line_items.apply(
    lambda r: (
        pd.notna(r["matched_service"])
        and (r["unit_price_cents"] != r["effective_unit_rate_cents"])
        and (not bool(r["weekend_applies"]))
        and (int(r["unit_price_cents"]) in _weekend_rate_options.get(r["matched_service"], set()))
    ), axis=1
)

# Volume-discount diagnostics.
_discount_rate_options = {}
for service, rows in volume_discounts.groupby("service"):
    base = int(base_rate_map[service])
    _discount_rate_options[service] = {
        half_up(Decimal(base) * (Decimal(100 - int(p)) / 100))
        for p in rows["discount_percent"]
    }
line_items["volume_discount_omitted"] = (
    (line_items["_discount_pct"] > 0)
    & (line_items["unit_price_cents"] == line_items["rate_after_weekend"])
    & (line_items["unit_price_cents"] != line_items["effective_unit_rate_cents"])
)
line_items["volume_discount_incorrectly_applied"] = line_items.apply(
    lambda r: (
        pd.notna(r["matched_service"])
        and (r["unit_price_cents"] != r["effective_unit_rate_cents"])
        and (
            ((int(r["_discount_pct"]) > 0) and not bool(r["volume_discount_omitted"]))
            or ((int(r["_discount_pct"]) == 0) and int(r["unit_price_cents"]) in _discount_rate_options.get(r["matched_service"], set()))
        )
    ), axis=1
)

print("Bundle applied:", int(line_items["bundle_applies"].sum()), "lines")
print("Premium applied:", int(line_items["premium_applies"].sum()), "lines")
print("Weekend uplift applied:", int(line_items["weekend_applies"].sum()), "lines")
print("Discount applied:", int((line_items["_discount_pct"] > 0).sum()), "lines")
print("Provisional expected lines:", int(line_items["expected_line_total_provisional"].sum()), "lines")

# Helpful debug view.
display(line_items[[
    "line_id", "invoice_id", "invoice_occurrence_key", "matched_service", "quantity",
    "billable_quantity", "base_rate_cents", "effective_unit_rate_cents",
    "expected_line_total_cents", "line_total_cents"
]].head())


Bundle applied: 540 lines
Premium applied: 182 lines
Weekend uplift applied: 209 lines
Discount applied: 1676 lines
Provisional expected lines: 17 lines


,line_id,invoice_id,invoice_occurrence_key,matched_service,quantity,billable_quantity,base_rate_cents,effective_unit_rate_cents,expected_line_total_cents,line_total_cents
0,H1-L00001-01,INV-H1-000001,INV-H1-000001::0,Routine Urologic Biopsy Procedure,1,1,226950.0,226950.0,226950.0,226950
1,H1-L00001-02,INV-H1-000001,INV-H1-000001::0,Intensive Ophthalmic Case Conference,13,13,21875.0,21875.0,284375.0,284375
2,H1-L00001-03,INV-H1-000001,INV-H1-000001::0,Ambulatory Infectious Home Visit,1,1,8225.0,8225.0,8225.0,8225
3,H1-L00001-04,INV-H1-000001,INV-H1-000001::0,Elective Pulmonary Nutritional Support,7,7,7450.0,7450.0,52150.0,52150
4,H1-L00001-05,INV-H1-000001,INV-H1-000001::0,Standard Otolaryngologic Radiotherapy Fraction,11,11,25250.0,25250.0,277750.0,277750


### Step 5 — Aggregate line totals to invoice totals

In [13]:
# Aggregate at invoice-occurrence level first, then collapse reused IDs to one prediction row.

# Billed line totals and contract expected totals per transaction occurrence.
occ_line = (
    line_items.groupby("invoice_occurrence_key").agg(
        expected_total_cents=("expected_line_total_cents", "sum"),
        any_provisional_expected=("expected_line_total_provisional", "any"),
        any_unmatched=("matched_service", lambda s: s.isna().any()),
        billed_line_total_sum_cents=("line_total_cents", "sum"),
    ).reset_index()
)
occ_line[["invoice_id", "occurrence_rank"]] = occ_line["invoice_occurrence_key"].str.split("::", expand=True)
occ_line["occurrence_rank"] = occ_line["occurrence_rank"].astype(int)

# Occurrence-specific line checks.
occ_flags = (
    line_items.groupby("invoice_occurrence_key")[LINE_CHECK_COLS + [
        "bundle_not_applied", "premium_omitted", "premium_incorrectly_applied",
        "volume_discount_omitted", "volume_discount_incorrectly_applied",
        "unit_price_mismatch", "weekend_uplift_omitted", "weekend_uplift_incorrectly_applied"
    ]].any().reset_index()
)

occ_invoice = invoices[
    ["invoice_id", "occurrence_rank", "invoice_date", "patient_id", "contract_number",
     "invoice_total_cents", "contract_number_mismatch", "duplicate_invoice_id"]
].copy()
occ_invoice["invoice_occurrence_key"] = (
    occ_invoice["invoice_id"].astype(str) + "::" + occ_invoice["occurrence_rank"].astype(str)
)

occ_summary = (
    occ_invoice.merge(occ_line, on=["invoice_occurrence_key", "invoice_id", "occurrence_rank"], how="left")
    .merge(occ_flags, on="invoice_occurrence_key", how="left")
)

# Explicit invoice-total arithmetic mismatch.
occ_summary["invoice_total_mismatch"] = (
    occ_summary["billed_line_total_sum_cents"] != occ_summary["invoice_total_cents"]
)

# Explicit contract-price mismatch exists when the contract-derived expected total differs from
# the invoice total and the amount is not merely an unresolved service placeholder.
# Duplicate invoice IDs: the development-set convention is to report the later invoice occurrence.
latest_rank = occ_summary.groupby("invoice_id")["occurrence_rank"].transform("max")
occ_summary["selected_occurrence"] = occ_summary["occurrence_rank"] == latest_rank

# Error categories are deliberately specific. Unknown service is a real audit flag and remains
# separate from the provisional expected-total calculation.
PRED_CAT_COLS = [
    "malformed_service_date", "service_date_out_of_window", "service_date_after_invoice_date",
    "wrong_unit_basis", "daily_cap_exceeded", "cross_invoice_duplicate", "exclusion_violation",
    "line_total_arithmetic", "bundle_not_applied", "unit_price_mismatch", "premium_omitted",
    "premium_incorrectly_applied", "weekend_uplift_omitted", "weekend_uplift_incorrectly_applied",
    "volume_discount_omitted", "volume_discount_incorrectly_applied", "invoice_total_mismatch",
    "contract_number_mismatch",
    "duplicate_invoice_id"
]

def occurrence_error_category(row):
    cats = [c for c in PRED_CAT_COLS if bool(row.get(c, False))]
    if bool(row.get("any_unmatched", False)):
        cats.append("unknown_service")
    return "|".join(dict.fromkeys(cats))

def occurrence_confidence(row):
    cats = [c for c in PRED_CAT_COLS if bool(row.get(c, False))]
    has_unknown = bool(row.get("any_unmatched", False))
    confidence = 0.95 if not cats and not has_unknown else 0.90
    if has_unknown:
        confidence = min(confidence, 0.35)
    if bool(row.get("duplicate_invoice_id", False)):
        confidence = min(confidence, 0.55)
    if bool(row.get("any_provisional_expected", False)):
        confidence = min(confidence, 0.55)
    if any(c in {"unit_price_mismatch", "premium_incorrectly_applied",
               "volume_discount_incorrectly_applied", "weekend_uplift_incorrectly_applied"}
           for c in cats):
        confidence = min(confidence, 0.80)
    if len(cats) >= 2:
        confidence = min(confidence, 0.75)
    return round(confidence, 2)

occ_summary["error_category"] = occ_summary.apply(occurrence_error_category, axis=1)
occ_summary["confidence"] = occ_summary.apply(occurrence_confidence, axis=1)
occ_summary["flagged"] = (occ_summary["error_category"] != "").astype(int)

# Collapse to one row per invoice ID. For duplicates this deliberately selects the later occurrence.
invoice_summary = occ_summary[occ_summary["selected_occurrence"]].copy()
invoice_summary = invoice_summary.sort_values(["invoice_id"]).reset_index(drop=True)

# Unmatched descriptions are flags, but the expected amount for those lines is provisional.
invoice_summary["billed_total_cents"] = invoice_summary["invoice_total_cents"]
invoice_summary["expected_total_cents"] = invoice_summary["expected_total_cents"].round().astype("Int64")

print("Flagged counts:", invoice_summary["flagged"].value_counts().to_dict())
print("\nConfidence distribution:")
print(invoice_summary["confidence"].value_counts().sort_index())
print("\nInvoices with provisional expected totals:", int(invoice_summary["any_provisional_expected"].sum()))


Flagged counts: {0: 855, 1: 58}

Confidence distribution:
confidence
0.35     11
0.55      9
0.75     28
0.90     10
0.95    855
Name: count, dtype: int64

Invoices with provisional expected totals: 15


### Step 6 — Evaluate against Hospital 1 development labels

Hospital 1 labels are used here to measure the final method and to document development-set calibration choices. They are not used by the runtime detector after the method is fixed. Because H1 is the labelled development set, any H1-tuned threshold or ambiguity rule is explicitly disclosed in the decision log and should be treated as development calibration rather than an unbiased holdout estimate.

Evaluation reports overall accuracy plus precision, recall and F1, then checks whether predicted error categories recover the documented label categories. Because the data are imbalanced, accuracy is reported alongside precision/recall rather than used alone.

The notebook also compares corrected totals against the labelled `expected_total_cents` and prints systematic residuals by failure type.


In [14]:
# Development-set evaluation — Hospital 1 only. Labels never feed back into the detector.

hospital_1_labels = pd.read_csv(REPO_ROOT / "labels" / "hospital_1_labels.csv")
check = invoice_summary.merge(
    hospital_1_labels, on="invoice_id", how="left", suffixes=("", "_label")
)

tp = int(((check["flagged"] == 1) & (check["is_erroneous"] == 1)).sum())
fp = int(((check["flagged"] == 1) & (check["is_erroneous"] == 0)).sum())
fn = int(((check["flagged"] == 0) & (check["is_erroneous"] == 1)).sum())
tn = int(((check["flagged"] == 0) & (check["is_erroneous"] == 0)).sum())

accuracy = (tp + tn) / len(check)
precision = tp / (tp + fp) if tp + fp else 0.0
recall = tp / (tp + fn) if tp + fn else 0.0
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

print(f"TP={tp}  FP={fp}  FN={fn}  TN={tn}")
print(f"accuracy={accuracy:.3f}  precision={precision:.3f}  recall={recall:.3f}  F1={f1:.3f}")

# Map internal category name to the label vocabulary where needed.
CATEGORY_MAP = {
    "unmatched_service_needs_review": "unknown_service",
    "unknown_service": "unknown_service",
    "exclusion_violation": "exclusion_window_violation",
    "weekend_uplift_omitted": "premium_omitted",
    "weekend_uplift_incorrectly_applied": "premium_incorrectly_applied",
}

def split_cats(value):
    if pd.isna(value) or not str(value).strip():
        return set()
    return {CATEGORY_MAP.get(x, x) for x in str(value).split("|")}

# Per-category detection recall/precision on the H1 dev labels.
label_categories = sorted({c for s in check["error_categories"] for c in split_cats(s)})
cat_rows = []
for cat in label_categories:
    label_pos = check["error_categories"].apply(lambda x: cat in split_cats(x))
    pred_pos = check["error_category"].apply(lambda x: cat in split_cats(x))
    tp_cat = int((label_pos & pred_pos).sum())
    fn_cat = int((label_pos & ~pred_pos).sum())
    fp_cat = int((~label_pos & pred_pos).sum())
    cat_rows.append({
        "category": cat,
        "label_invoices": int(label_pos.sum()),
        "detected": tp_cat,
        "missed": fn_cat,
        "category_recall": tp_cat / int(label_pos.sum()) if label_pos.sum() else 0.0,
        "category_precision": tp_cat / (tp_cat + fp_cat) if tp_cat + fp_cat else 0.0,
    })
category_metrics = pd.DataFrame(cat_rows).sort_values(["category_recall", "category"])
print("\nPer-category development performance:")
display(category_metrics)

# Expected total accuracy. Keep provisional rows visible instead of hiding them.
check["expected_total_exact"] = check["expected_total_cents"] == check["expected_total_cents_label"]
print(
    f"\nExact expected_total_cents match: {check['expected_total_exact'].mean():.1%} overall"
)
print(
    f"Exact expected_total_cents match on labelled-correct invoices: "
    f"{check.loc[check['is_erroneous'] == 0, 'expected_total_exact'].mean():.1%}"
)
print(
    f"Exact expected_total_cents match on labelled-erroneous invoices: "
    f"{check.loc[check['is_erroneous'] == 1, 'expected_total_exact'].mean():.1%}"
)

# Systematic residuals rather than a list of individual misses.
residuals = check[~check["expected_total_exact"]].copy()
if residuals.empty:
    print("\nNo labelled expected-total residuals remain.")
else:
    def residual_type(row):
        label = split_cats(row["error_categories"]); pred = split_cats(row["error_category"])
        if "daily_cap_exceeded" in label:
            return "daily-cap correction"
        if "duplicate_invoice_id" in label:
            return "duplicate-ID occurrence selection"
        if "exclusion_window_violation" in label:
            return "exclusion-window handling"
        if "unknown_service" in label:
            return "unknown-service provisional pricing"
        if "malformed_service_date" in label:
            return "malformed-date pricing"
        if "volume_discount" in label:
            return "volume-discount pricing"
        return "other contract-pricing residual"
    residuals["failure_type"] = residuals.apply(residual_type, axis=1)
    print("\nExpected-total residuals by failure type:")
    # Report the number of residual invoices and the magnitude of the discrepancy.
    # These are development-set diagnostics, not changes to the contract rules.
    residuals["abs_difference_cents"] = (
        residuals["expected_total_cents_label"] - residuals["expected_total_cents"]
    ).abs()
    display(
        residuals.groupby("failure_type").agg(
            invoices=("invoice_id", "count"),
            mean_abs_difference_cents=("abs_difference_cents", "mean"),
            total_abs_difference_cents=("abs_difference_cents", "sum"),
        ).reset_index().sort_values("invoices", ascending=False)
    )

# H1 calibration note for confidence: these are policy scores, not statistical probabilities.
check["prediction_correct"] = check["flagged"] == check["is_erroneous"]
print("\nMean stated confidence:")
print(check.groupby("prediction_correct")["confidence"].mean().rename(index={True: "correct", False: "incorrect"}))

# Confidence-bucket calibration check. With only 913 H1 invoices, treat these as descriptive
# development diagnostics rather than statistical calibration claims.
confidence_bins = pd.cut(
    check["confidence"],
    bins=[0, 0.5, 0.7, 0.85, 0.95, 1.000001],
    include_lowest=True,
)
confidence_metrics = check.assign(confidence_bucket=confidence_bins).groupby("confidence_bucket", observed=False).agg(
    invoices=("invoice_id", "count"),
    prediction_accuracy=("prediction_correct", "mean"),
    erroneous_invoices=("is_erroneous", "sum"),
).reset_index()
print("\nConfidence-bucket development check:")
display(confidence_metrics)

print("\nConfidence is heuristic audit certainty, not a calibrated probability of correctness.")


TP=58  FP=0  FN=0  TN=855
accuracy=1.000  precision=1.000  recall=1.000  F1=1.000

Per-category development performance:


,category,label_invoices,detected,missed,category_recall,category_precision
14,unknown_service,12,11,1,0.916667,1.000000
0,bundle_not_applied,5,5,0,1.000000,1.000000
1,contract_number_mismatch,5,5,0,1.000000,1.000000
2,cross_invoice_duplicate,4,4,0,1.000000,1.000000
3,daily_cap_exceeded,4,4,0,1.000000,1.000000
4,duplicate_invoice_id,5,5,0,1.000000,1.000000
5,exclusion_window_violation,4,4,0,1.000000,1.000000
6,invoice_total_mismatch,6,6,0,1.000000,1.000000
7,line_total_arithmetic,6,6,0,1.000000,1.000000
8,malformed_service_date,6,6,0,1.000000,1.000000



Exact expected_total_cents match: 99.5% overall
Exact expected_total_cents match on labelled-correct invoices: 100.0%
Exact expected_total_cents match on labelled-erroneous invoices: 91.4%

Expected-total residuals by failure type:


,failure_type,invoices,mean_abs_difference_cents,total_abs_difference_cents
0,daily-cap correction,4,76118.75,304475
1,unknown-service provisional pricing,1,43650.0,43650



Mean stated confidence:
prediction_correct
correct    0.932147
Name: confidence, dtype: float64

Confidence-bucket development check:


,confidence_bucket,invoices,prediction_accuracy,erroneous_invoices
0,"(-0.001, 0.5]",11,1.0,11
1,"(0.5, 0.7]",9,1.0,9
2,"(0.7, 0.85]",28,1.0,28
3,"(0.85, 0.95]",865,1.0,10
4,"(0.95, 1.0]",0,NaN,0



Confidence is heuristic audit certainty, not a calibrated probability of correctness.


### Hospital 1 decision log

**Uncertainty handling.** Unmatched/ambiguous service descriptions are flagged for review with low confidence rather than silently treated as valid. Billed price is used only as secondary evidence when text and structural constraints leave a genuinely narrow candidate set.

**Corrected totals.** Exclusion-window `service_a` lines and later cross-invoice duplicate lines are treated as non-billable in the corrected expected total. Daily quantity caps reduce billable quantity before multiplication. Unknown services or malformed dates retain their billed line amount provisionally and are marked low-confidence because a defensible contract rate cannot be established automatically.

**Development calibration.** The H1 development labels were used to test threshold/ambiguity choices while building the method. The runtime detector does not read the label file. In particular, the reused-invoice-ID handling was validated against H1 and then encoded as a deterministic occurrence rule; this calibration is disclosed rather than presented as an unbiased test result.

**Development-set discrepancy.** Four H1 labelled daily-cap cases do not reproduce the labelled corrected total when the written Section 8 cap is applied literally. The notebook keeps the contract-faithful cap interpretation instead of hard-coding the development labels, and reports the resulting residuals explicitly. This is a methodology choice to revisit only if the task owner clarifies the intended cap convention.

**Category vocabulary.** The detector keeps precise internal categories such as `weekend_uplift_omitted`; the H1 evaluation maps those to the broader label vocabulary where appropriate. `pricing_mismatch` is no longer used as a catch-all prediction category.

**Confidence.** Confidence values are deterministic audit-policy scores reflecting evidence quality and ambiguity; they are not probabilities. The notebook reports confidence buckets descriptively on H1 rather than claiming statistical calibration from the development set alone.


### Step 7 — Final Hospital 1 development output

This is generated in the exact repository submission schema. `hospital_1_dev_predictions.csv` is a calibration artifact only; Hospital 1 is not the scored holdout.

Review cases remain flagged with low confidence and a provisional expected total rather than being silently classified as correct.

In [15]:
# Hospital 1 development output in the exact submission-template schema.

submission_hospital_1 = invoice_summary.rename(
    columns={"error_category": "error_category"}
)[
    ["invoice_id", "flagged", "error_category", "expected_total_cents", "billed_total_cents", "confidence"]
].copy()

# Submission schema sanity checks.
required_columns = [
    "invoice_id", "flagged", "error_category",
    "expected_total_cents", "billed_total_cents", "confidence"
]
assert submission_hospital_1.columns.tolist() == required_columns
assert submission_hospital_1["invoice_id"].is_unique
assert submission_hospital_1["flagged"].isin([0, 1]).all()
assert submission_hospital_1["confidence"].between(0, 1).all()

output_path = REPO_ROOT / "hospital_1_dev_predictions.csv"
submission_hospital_1.to_csv(output_path, index=False)
print(f"Wrote {len(submission_hospital_1)} rows to {output_path}")
display(submission_hospital_1.head(10))


Wrote 913 rows to /mnt/data/insurance_repo/insurance_auditing-main/hospital_1_dev_predictions.csv


,invoice_id,flagged,error_category,expected_total_cents,billed_total_cents,confidence
0,INV-H1-000001,0,,2475525,2475525,0.95
1,INV-H1-000002,1,service_date_after_invoice_date,2588836,2588836,0.90
2,INV-H1-000003,0,,5777580,5777580,0.95
3,INV-H1-000004,1,unit_price_mismatch|volume_discount_incorrectl...,924200,1260398,0.75
4,INV-H1-000005,0,,573375,573375,0.95
5,INV-H1-000006,0,,1746150,1746150,0.95
6,INV-H1-000007,0,,2958400,2958400,0.95
7,INV-H1-000008,0,,798790,798790,0.95
8,INV-H1-000009,0,,2637550,2637550,0.95
9,INV-H1-000010,0,,2276950,2276950,0.95


**Hospital 1 status**

The revised pipeline now separates four things that were previously conflated: service identity, contract checks, corrected pricing, and confidence.

Key safeguards carried forward to later hospitals:

* unmatched/ambiguous service descriptions are **flagged** and kept at low confidence rather than being treated as correct;
* price is secondary evidence for service matching, never the sole reason to select a service;
* reused invoice IDs are processed by transaction occurrence instead of merging all lines together;
* exclusion windows remove the contract-defined excluded service (`service_a`) from the corrected total;
* daily caps reduce billable quantity before line-total aggregation;
* line arithmetic and invoice-total arithmetic are explicit checks;
* corrected totals remain provisional when the contract basis cannot be established;
* Hospital 1 labels are used only in the development evaluation section.

The next step for other hospitals would be to re-extract their contract rules and keep the same audit/evaluation framework, without tuning directly to their unseen labels.